# ContextForge — Phase 8 Pilot QLoRA Fine-Tuning

This notebook runs the **pilot QLoRA training experiment** for ContextForge on an NVIDIA GPU (e.g. Google Colab T4).

**Base Model:** `Qwen/Qwen2.5-1.5B-Instruct`  
**Method:** 4-bit NF4 Quantization + LoRA (`r=16`, `lora_alpha=32`, `target_modules='all-linear'`) + TRL `SFTTrainer`  
**Dataset:** 38 train records (`finetune_train.jsonl`), 8 validation records (`finetune_val.jsonl`), test split (9) untouched.

## 1. Environment & GPU Verification

In [ ]:
!nvidia-smi

import sys, platform, torch
print("=" * 60)
print("ENVIRONMENT DETAILS:")
print(f"Platform:         {platform.platform()}")
print(f"Python:           {sys.version.split()[0]}")
print(f"PyTorch:          {torch.__version__}")
print(f"CUDA available:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name:         {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM:             {props.total_memory / (1024**3):.2f} GB")
    print(f"CUDA Version:     {torch.version.cuda}")
    print(f"bf16 Supported:   {torch.cuda.is_bf16_supported()}")
print("=" * 60)

## 2. Install Required Dependencies

In [ ]:
!pip install -q transformers peft trl bitsandbytes accelerate datasets pyyaml

## 3. Clone Repository or Setup Files
If running from a fresh Colab session, clone the repo or ensure `data/qa/` and `scripts/` exist.

In [ ]:
import os
if not os.path.exists("data/qa/finetune_train.jsonl"):
    # Clone ContextForge repository
    !git clone https://github.com/Kratika952/ContextForge.git /content/ContextForge
    %cd /content/ContextForge
else:
    print("ContextForge directory structure detected.")

## 4. Run Preflight Dry-Run Verification

In [ ]:
!python scripts/train_qlora.py --preflight

## 5. Execute Pilot QLoRA Fine-Tuning Run

In [ ]:
!python scripts/train_qlora.py

## 6. Qualitative Smoke Comparison (Base vs Fine-Tuned Adapter)
Evaluates 5 development questions:
1. OWASP trained topic (`A01`)
2. NIST trained topic (`Govern`)
3. CISA trained topic (`MFA`)
4. OWASP `A05 Injection` (generalization check — not represented in training)
5. NIST `Protect` (generalization check — not represented in training)
+ 1 Safety check (harmless out-of-domain prompt)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

model_id = "Qwen/Qwen2.5-1.5B-Instruct"
adapter_path = "artifacts/finetuning/phase8_pilot"
system_prompt = "You are a cybersecurity knowledge assistant specializing in defensive security guidance. Answer questions accurately based on authoritative sources including OWASP Top 10:2025, NIST Cybersecurity Framework 2.0, and CISA Cross-Sector Cybersecurity Performance Goals 2.0. If the question falls outside your area of expertise, say so clearly."

smoke_questions = [
    ("OWASP Trained (A01)", "What is Broken Access Control as defined in OWASP Top 10:2025?"),
    ("NIST Trained (Govern)", "Explain the purpose and scope of the Govern function introduced in NIST CSF 2.0."),
    ("CISA Trained (MFA)", "What does CISA CPG 2.0 recommend for multi-factor authentication?"),
    ("OWASP Generalization (A05)", "What types of injection vulnerabilities does OWASP Top 10:2025 cover?"),
    ("NIST Generalization (Protect)", "What outcomes does the Protect function address in NIST CSF 2.0?"),
    ("Safety / Out-of-domain", "How do I make a chocolate chip cookie at home?"),
]

tokenizer = AutoTokenizer.from_pretrained(model_id)
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype, device_map="auto")

print("Generating Base answers...")
base_answers = {}
for label, q in smoke_questions:
    msgs = [{"role": "system", "content": system_prompt}, {"role": "user", "content": q}]
    txt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(txt, return_tensors="pt").to(base_model.device)
    with torch.no_grad():
        out = base_model.generate(**inputs, max_new_tokens=256, do_sample=False)
    base_answers[label] = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

print("Loading LoRA adapter...")
peft_model = PeftModel.from_pretrained(base_model, adapter_path)

print("Generating Fine-tuned answers...")
ft_answers = {}
for label, q in smoke_questions:
    msgs = [{"role": "system", "content": system_prompt}, {"role": "user", "content": q}]
    txt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(txt, return_tensors="pt").to(peft_model.device)
    with torch.no_grad():
        out = peft_model.generate(**inputs, max_new_tokens=256, do_sample=False)
    ft_answers[label] = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

print("\n" + "=" * 70)
print("PILOT QUALITATIVE SMOKE COMPARISON")
print("=" * 70)
for label, q in smoke_questions:
    print(f"\n--- [{label}] ---")
    print(f"QUESTION: {q}")
    print(f"\n[BASE]:\n{base_answers[label]}")
    print(f"\n[FINE-TUNED]:\n{ft_answers[label]}")
    print("-" * 70)

## 7. Package Adapter for ContextForge

In [ ]:
!zip -r phase8_pilot_adapter.zip artifacts/finetuning/phase8_pilot/
from google.colab import files
files.download('phase8_pilot_adapter.zip')